# 检索、引用与模型回答：跟着证据做判断

做完本实验，你应能回答三个问题：找到文档与把文档排在前面有什么区别？为什么引文抄对了仍可能答错？怎么证明下一次回答确实读了保存的记忆？

前两节**读取已存的真实模型报告，并重新计算评分**，不会下载或重新调用模型。后两节实际执行 SQLite 记忆与构造数据统计。完整模型重跑命令、依赖与边界见 [README](README.md)。先安装仓库的 `requirements-dev.lock`，选择同一个虚拟环境内核，再按顺序运行。


In [1]:
from pathlib import Path
import json
import sys
import tempfile
from statistics import mean
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/run_python.py").is_file())
for pattern in ("10-Knowledge/*/05-code/*/src", "20-Projects/*/src"):
    for source in ROOT.glob(pattern):
        if str(source) not in sys.path:
            sys.path.insert(0, str(source))
PROJECT = ROOT / "20-Projects/learning-workbench"
MODELS = PROJECT / "artifacts/real-models"

def table(headers, rows):
    # Small summaries stay readable; raw reports remain available in artifacts/.
    def clean(value):
        return str(value).replace("|", r"\|").replace("\n", " ")
    lines = ["| " + " | ".join(headers) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(clean(v) for v in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


## 1. 找全与排前是两个目标

令 $G_q$ 是问题 $q$ 的相关文档 ID 集合，$R_{q,3}$ 是返回的前 3 个文档。这里按文档评分，同一文档的多个片段不重复计数。

$$\mathrm{Recall@3}(q)=\frac{|G_q\cap R_{q,3}|}{|G_q|},\qquad
\mathrm{RR}(q)=\begin{cases}1/r_q,&\text{前 3 位中找到相关文档}\\0,&\text{否则}\end{cases}$$

$r_q$ 是首个相关文档的位置，从 1 开始数。MRR 是各题 RR 的平均值。本数据的每题都有相关文档；若 $G_q$ 为空，Recall 分母为零，应另作无答案任务评分。

先看各模式均值，再看差异来自哪道题。这 6 道英文构造题不足以推断中文业务上的检索效果。


In [2]:
retrieval = json.loads((MODELS / "retrieval.json").read_text(encoding="utf-8"))
trials = retrieval["trials"]
summary = []
for mode in dict.fromkeys(row["mode"] for row in trials):
    group = [row for row in trials if row["mode"] == mode]
    # Recompute from ranking and gold IDs, not from the report's score fields.
    recalls = [len(set(row["retrieved"]) & set(row["relevant"])) / len(row["relevant"])
               for row in group]
    rr = [next((1 / rank for rank, doc_id in enumerate(row["retrieved"], 1)
                if doc_id in row["relevant"]), 0) for row in group]
    assert all(abs(score - row["recall_at_3"]) < 1e-12 for score, row in zip(recalls, group))
    assert all(abs(score - row["reciprocal_rank"]) < 1e-12 for score, row in zip(rr, group))
    summary.append([mode, len(group), f"{mean(recalls):.3f}", f"{mean(rr):.3f}"])
table(["模式", "题数", "平均 Recall@3", "MRR（前3位）"], summary)


| 模式 | 题数 | 平均 Recall@3 | MRR（前3位） |
| --- | --- | --- | --- |
| bm25 | 6 | 0.833 | 0.833 |
| dense | 6 | 1.000 | 1.000 |
| hybrid | 6 | 1.000 | 0.889 |
| hybrid-rerank | 6 | 1.000 | 1.000 |

In [3]:
q3 = [row for row in trials if row["id"] == "q3"]
print("问题：", q3[0]["query"], "；相关文档：", q3[0]["relevant"])
table(["模式", "前3个文档，顺序即排名", "Recall@3", "RR"],
      [[row["mode"], " → ".join(row["retrieved"]), row["recall_at_3"],
        f'{row["reciprocal_rank"]:.3f}'] for row in q3])


问题： The internet connection stopped working. ；相关文档： ['network']


| 模式 | 前3个文档，顺序即排名 | Recall@3 | RR |
| --- | --- | --- | --- |
| bm25 | memory → fan-v1 → rain | 0.0 | 0.000 |
| dense | network → fan-v1 → fan-v2 | 1.0 | 1.000 |
| hybrid | fan-v1 → memory → network | 1.0 | 0.333 |
| hybrid-rerank | network → battery → memory | 1.0 | 1.000 |

这里应看到：BM25 没找到 `network`；Hybrid 找到了却排第 3，Recall@3 = 1、RR = 1/3；重排后变成第 1。Dense 在这题本来就是第 1，因此“融合后总会更好”不成立。

**先预测再修改**：若 `network` 改到第 2 位，Recall@3 会变吗，RR 会变成多少？答案是 Recall 不变、RR = 1/2。重排改善排序；若候选里根本没有 `network`，重排不能凭空找回它。实现见 [retrieval_comparison](src/learning_workbench/retrieval.py)。


## 2. 引用存在，主张却可能答非所问

检查保存结果中的 `g2`。请先看问题，再逐条比对 `text` 和 `quote`，最后看独立审阅标签。`reason` 是模型自己的解释，不是评分结果。


In [4]:
generation = json.loads((MODELS / "generation.json").read_text(encoding="utf-8"))
review = json.loads((MODELS / "generation-review.json").read_text(encoding="utf-8"))
g2 = next(row for row in generation["trials"] if row["id"] == "g2")
print("问题：", g2["question"])
print("模型自己的 reason：", g2["answer"]["reason"])
table(["主张 text", "所引原文 quote"],
      [[claim["text"], claim["quote"]] for claim in g2["answer"]["claims"]])
table(["题号", "引文可定位", "引文支持主张", "回答问题"],
      [[row["id"], row["citation_identity"], row["semantic_support"], row["answers_question"]]
       for row in review["trials"]])


问题： FAN-01: What is the DEMO-A temperature limit?
模型自己的 reason： The text directly states the temperature limit for FAN-01 as 68°C.


| 主张 text | 所引原文 quote |
| --- | --- |
| Before inspecting a cooling fan, disconnect power | Version 2 changes the DEMO-A temperature limit to 68 C. |
| Before inspecting a cooling fan, disconnect power | Version 2 changes the DEMO-A temperature limit to 68 C. |

| 题号 | 引文可定位 | 引文支持主张 | 回答问题 |
| --- | --- | --- | --- |
| g1 | not assessable | not assessable | False |
| g2 | True | False | False |
| g3 | not assessable | not assessable | False |
| g4 | not assessable | not assessable | False |

问题问 DEMO-A 温度上限，`reason` 提到了 68°C，但正式 `claims` 重复写“先断电”，所配引文却只讲温度。引文确实来自资料，不能推出这条配对主张，也没有在正式主张中回答温度问题。因此 `citation_identity_valid=true`、`abstention_correct=true` 都不能等同于完整答案正确。

其他 3 题的 `not assessable` 表示输出未通过 JSON 契约，没有可接纳的主张可供引用判断；不能把格式失败计作成功拒答。这里的审阅是作者对这几个固定输出的标签，不是专家标注基准。对新问题还需要独立审阅。

读代码时从 [generated_answer](src/learning_workbench/retrieval.py) 找两个位置：查询先绑定 `tenant/product/version`；生成后只确定性检查引用 ID 与原文字串。语义支持尚不能由子串匹配解决。


## 3. 真正关闭再打开记忆库

仅有“无记忆/有记忆”的分数不够，先做一次可观察的写入与读取。第一段明确说“以后”，写入长期偏好；关闭对象后，在第二个对象中打开同一个 SQLite 文件；当前指令再覆盖旧偏好。这个实验验证持久化到数据库，不等于主机故障恢复测试。


In [5]:
from learning_workbench.memory import MemoryAssistant

with tempfile.TemporaryDirectory() as directory:
    db = Path(directory) / "learner-memory.sqlite"
    first_session = MemoryAssistant(db)
    try:
        saved = first_session.remember("我以后希望用表格回答", subject="alice",
                                       source="chat:session1:turn1", now=1)
        assert saved["accepted"]
    finally:
        first_session.close()

    second_session = MemoryAssistant(db)
    try:
        remembered = second_session.reply("解释State和Memory", subject="alice", now=2)
        overridden = second_session.reply("这次用段落解释State和Memory", subject="alice", now=2)
        rejected = second_session.remember("我以后不要用表格", subject="alice",
                                          source="chat:session2:turn2", now=2)
        assert remembered["format"] == "table"
        assert overridden["format"] == "paragraph"
        assert rejected["accepted"] is False
        print("重开后实际召回：", remembered["context"]["selected_memory"])
        display(Markdown(remembered["text"]))
        print("当前要求覆盖：", overridden["text"])
        print("不支持的否定输入：", rejected)
        second_session.store.forget("alice", "answer_format")
        assert second_session.reply("解释State和Memory", subject="alice", now=3)["format"] == "paragraph"
    finally:
        second_session.close()


重开后实际召回： {'subject': 'alice', 'key': 'answer_format', 'value': 'table', 'source': 'chat:session1:turn1', 'updated_at': 1.0, 'expires_at': None, 'version': 1}
当前要求覆盖： State是当前任务事实；Memory是供以后任务检索的信息。
不支持的否定输入： {'accepted': False, 'reason': 'unsupported format expression; use one complete affirmative template'}


| 概念 | 含义 |
| --- | --- |
| State | 当前任务事实 |
| Memory | 供以后任务检索的信息 |

保存下来的内容是格式偏好，不是固定回答全文。`subject` 绑定 Alice，`source` 记录输入出处，`version` 帮助检测覆盖冲突。上面临时数据库在单元结束后清理；希望长期观察，可改成自己的运行目录。

这个提取器只识别单一肯定格式。“不要用表格”不能被反向存成喜欢表格，因此明确拒绝解析；这也不会自动删除旧偏好，删除使用 `forget`。当前请求若是否定或多种格式，同样需要明确处理错误，不能悄悄回退到旧偏好。实际系统还要处理引语、多人、条件与冲突等语言情况。

下面重新运行 8 个固定任务。比较 `m0` 与 `m1`，再找出主体不同、当前要求覆盖、过期、删除这几种情况对应的行。


In [6]:
from learning_workbench.cli import read_jsonl
from learning_workbench.memory import evaluate_memory

with tempfile.TemporaryDirectory() as directory:
    memory = evaluate_memory(read_jsonl("memory-tasks.jsonl"), directory)
assert all(row["correct"] for row in memory)
table(["任务", "无记忆格式", "有记忆格式", "预期", "有记忆判定"],
      [[row["id"], row["answers"]["without_memory"]["format"],
        row["answers"]["with_memory"]["format"], row["expected"], row["correct"]]
       for row in memory])
print("无记忆/有记忆正确数：", sum(row["baseline_correct"] for row in memory),
      "/", sum(row["correct"] for row in memory), "；题目数：", len(memory))


无记忆/有记忆正确数： 6 / 8 ；题目数： 8


| 任务 | 无记忆格式 | 有记忆格式 | 预期 | 有记忆判定 |
| --- | --- | --- | --- | --- |
| m0 | paragraph | table | table | True |
| m1 | paragraph | paragraph | paragraph | True |
| m2 | paragraph | paragraph | paragraph | True |
| m3 | paragraph | paragraph | paragraph | True |
| m4 | paragraph | paragraph | paragraph | True |
| m5 | paragraph | paragraph | paragraph | True |
| m6 | paragraph | paragraph | paragraph | True |
| m7 | paragraph | bullets | bullets | True |

## 4. 重复 Trial 不能当作新 Task

统计演示有 $T=12$ 道构造题，每题 5 次 Trial。同一道题的改进值共享题目效应，不能把 60 次试验当成 60 道独立题。这里先对每题取配对差值的均值，再对题目平均：

$$\Delta_t=\frac{1}{n_t}\sum_{j=1}^{n_t}(s^{candidate}_{tj}-s^{baseline}_{tj}),\qquad
\widehat\Delta=\frac{1}{T}\sum_{t=1}^{T}\Delta_t.$$

`Task bootstrap` 每次有放回抽取 12 个**整题差值**，重算均值，共重复 1000 次；取经验分位数作区间。右侧的 `naive Trial` 对照把每次 Trial 差值打散，会掩盖题目间差异。两者在本例每题试验次数相同；次数不同时还需明确是每题等权还是每次试验等权，不能混用估计目标。


In [7]:
from learning_workbench.experiments import statistics_experiment

experiment = statistics_experiment()
report = experiment["report"]
table(["抽样单位", "95%经验区间", "宽度"],
      [[name, [round(x, 4) for x in report[key]],
        round(report[key][1] - report[key][0], 4)]
       for name, key in [("Task", "task_bootstrap_95"), ("错误地打散 Trial", "naive_trial_bootstrap_95")]])
print("平均配对差：", round(report["mean_paired_difference"], 4),
      "；Task 数：", report["task_count"], "；每题 Trial 数：", report["trial_counts"])
assert report["task_bootstrap_95"][1] - report["task_bootstrap_95"][0] > (
    report["naive_trial_bootstrap_95"][1] - report["naive_trial_bootstrap_95"][0])


平均配对差： 0.0625 ；Task 数： 12 ；每题 Trial 数： [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]


| 抽样单位 | 95%经验区间 | 宽度 |
| --- | --- | --- |
| Task | [-0.05, 0.175] | 0.225 |
| 错误地打散 Trial | [0.0025, 0.1225] | 0.12 |

这组数据的前 7 题人为增加 0.25，后 5 题人为减少 0.20，所以平均配对差为 `(7×0.25−5×0.20)/12 = 0.0625`。这只是构造效应，既不是模型实测收益，也不能用更窄的错误区间宣称稳定提升。

完成后，用自己的话解释：

| 自查问题 | 应抓住的判断 |
| --- | --- |
| Recall=1 而 RR=1/3，应该优先修什么？ | 相关文档已召回，先检查排序；别把它当缺少资料 |
| 引文是原文，答案为什么仍可能错误？ | 引文可能不支持该主张，主张也可能没回答问题 |
| 记忆写入后，下次回复没变化，先查哪里？ | 看实际召回记录、主体、过期时间、当前指令，不先假定是模型问题 |
| 同一题多跑几次，为何不能代替增加新题？ | 重复试验仍共享题目难度与效应，题目覆盖没有扩大 |

继续实践可从 [规划实验](src/learning_workbench/planning.py) 修改一个证据版本，或按 [README](README.md) 接入模型重新生成报告。保存新报告到自己的 `.runs/` 目录，与仓库中既有结果分别比较。
